# Optional LSTM Baseline with Shared MFCC Sequence Cache


## 1. Load LSTM-Ready Cache


In [ ]:
# Purpose: Loads the optional LSTM-ready cache. This notebook does not scan audio folders,
# make a new split, or call Librosa MFCC extraction.
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
EPOCHS = 30
BATCH_SIZE = 64
THRESHOLD = 0.5


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_CLASS_WEIGHT_PATH = PROJECT_ROOT / "outputs" / "shared" / "class_weights.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm_optional"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the analysis notebook, then 00_LSTM_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train_scaled = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_meta = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation_scaled = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_meta = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
mfcc_mean = np.load(CACHE_DIR / "mfcc_mean.npy")
mfcc_std = np.load(CACHE_DIR / "mfcc_std.npy")
with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS



print("Loaded LSTM-ready cache:", CACHE_DIR)
print("X_train shape:", X_train_scaled.shape)
print("X_validation shape:", X_validation_scaled.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Validation Experiments


In [ ]:
# Purpose: Trains optional LSTM baseline candidates on train+validation only. The test
# cache exists for the final selected notebook, but no test prediction is made here.
RUN_LSTM_BASELINE = False
experiments = [
    {"name": "lstm_64_dropout_030", "units": 64, "dropout": 0.30, "learning_rate": 0.001},
    {"name": "lstm_32_dropout_025", "units": 32, "dropout": 0.25, "learning_rate": 0.001},
]


def build_lstm_model(config):
    model = Sequential()
    model.add(Input(shape=X_train_scaled.shape[1:]))
    model.add(LSTM(int(config["units"])))
    model.add(Dropout(float(config["dropout"])))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=float(config["learning_rate"]))
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


if RUN_LSTM_BASELINE:
    validation_results = []
    histories = {}
    for experiment in experiments:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(RANDOM_STATE)
        model = build_lstm_model(experiment)
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
        history = model.fit(
            X_train_scaled,
            y_train,
            validation_data=(X_validation_scaled, y_validation),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=CLASS_WEIGHTS,
            callbacks=callbacks,
            verbose=2,
        )
        probability = model.predict(X_validation_scaled, batch_size=BATCH_SIZE, verbose=0).ravel()
        pred = (probability >= THRESHOLD).astype(int)
        validation_results.append(
            {
                "model": "Optional LSTM",
                "variant": experiment["name"],
                "accuracy": accuracy_score(y_validation, pred),
                "precision": precision_score(y_validation, pred, zero_division=0),
                "recall": recall_score(y_validation, pred, zero_division=0),
                "f1": f1_score(y_validation, pred, zero_division=0),
                "best_val_loss": float(np.min(history.history["val_loss"])),
                "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
            }
        )
        histories[experiment["name"]] = history.history
    validation_results_df = pd.DataFrame(validation_results).sort_values("f1", ascending=False)
    validation_results_df.to_csv(TABLES_DIR / "lstm_validation_experiments.csv", index=False)
    display(validation_results_df)
else:
    print("RUN_LSTM_BASELINE is False. Set it to True when ready to run optional LSTM validation experiments.")


## 3. Reproducibility Checks


In [ ]:
checks = {
    "optional_model": True,
    "loads_lstm_ready_cache": str(CACHE_DIR),
    "does_not_rescan_audio": True,
    "does_not_create_split": True,
    "does_not_extract_mfcc": True,
    "input_representation": "T x 40 MFCC sequence",
    "uses_shared_class_weights": str(SHARED_CLASS_WEIGHT_PATH),
    "test_metrics_computed_here": False,
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
